In [ ]:
import sys, torch

if not torch.cuda.is_available():
    print("=" * 60)
    print("  ERROR: No GPU detected!")
    print("  This notebook requires a GPU runtime.")
    print("  Go to Runtime > Change runtime type > Hardware accelerator: GPU")
    print("=" * 60)
    sys.exit(1)

print(f"GPU OK: {torch.cuda.get_device_name(0)}  ({torch.cuda.device_count()} device(s))")
print(f"Torch CUDA: {torch.version.cuda}")

# Stage-wise BC + PPO Training for BomberGame

This notebook trains a hybrid neural + codex agent using stage-wise RL.

Based on the RL stage-wise patch by BFC. Uses GPU acceleration.

**Recommended reading**: [config.md](https://github.com/BFCmath/AIC-GDGoC-2026/blob/main/docs/patches/config.md)

---
**Workflow**:
1. Clone repo + install deps
2. Behavior Cloning (BC) pretraining
3. Stage-wise PPO training (stages 0 → 7)
4. Export agent
5. Benchmark against codex agents
6. (Optional) Late-stage finetune

In [ ]:
# Cell 1: Clone repo, apply patches (if needed), install deps
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/BFCmath/AIC-GDGoC-2026.git"
REPO_DIR = Path("/content/AIC-GDGoC-2026")

if "google.colab" in sys.modules:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(str(REPO_DIR))
else:
    REPO_DIR = Path.cwd()

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Repo:", REPO_DIR)

# Apply RL training patches if not already applied.
# Sentinel: stage_train_bc_ppo.py is only created by the patch.
if not (REPO_DIR / "scripts" / "participant" / "stage_train_bc_ppo.py").exists():
    PATCHES_DIR = REPO_DIR / "docs" / "patches"
    if PATCHES_DIR.exists():
        p1 = PATCHES_DIR / "rl_training_patch (1).diff"
        if p1.exists():
            print(f"Applying {p1.name}...")
            subprocess.run(["patch", "-p1", "-i", str(p1)], cwd=REPO_DIR, check=True)
        p2 = PATCHES_DIR / "rl_stagewise_patch.diff"
        if p2.exists():
            print(f"Applying {p2.name}...")
            subprocess.run(["patch", "-p2", "-i", str(p2)], cwd=REPO_DIR, check=True)
        print("Patches applied.")
    else:
        print("ERROR: patches not found at docs/patches/. Cannot train.")
        sys.exit(1)
else:
    print("Repo already patched, skipping patch step.")

# Install deps filtering out torch/tf/sb3 (Colab has CUDA torch pre-installed)
reqs = Path("requirements.txt").read_text().splitlines()
reqs = [r for r in reqs if r and not r.startswith("#")
        and not r.startswith("torch")
        and not r.startswith("tensorflow")
        and not r.startswith("stable-baselines3")]
if reqs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + reqs, check=True)
print("Dependencies installed.")

def stream_run(cmd, check=True):
    """Run cmd with real-time stdout/stderr streaming."""
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if check and proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    return proc.returncode



## Configuration

The recommended config is:
- BC: 80 matches, 40 epochs
- PPO: 900 total updates, 16 envs, horizon 128
- Eval: 12 matches per probe
- Profile: `kaggle_medium`

Edit the cell below to customize.

In [ ]:
# Cell 2: Configuration — edit these values as needed
PROFILE = "kaggle_medium"       # smoke | kaggle_medium | kaggle_long
DEVICE = "cuda"                  # cuda or cpu
EVAL_MATCHES = 12                # matches per evaluation probe
TORCH_THREADS = 1                # 1 for Colab to avoid oversubscription
EXPORT_DIR = "exports/stagewise_hybrid_agent"
SEED = 86

# Optional: resume from a specific stage (set START_STAGE > 0 with SKIP_BC=True)
SKIP_BC = False                  # set True to skip BC and resume from checkpoint
START_STAGE = 0                   # resume from this stage (0-7)
END_STAGE = 7                     # train up to this stage (0-7)

print(f"Profile: {PROFILE}")
print(f"Device: {DEVICE}")
print(f"Export dir: {EXPORT_DIR}")
print(f"Stages: {START_STAGE} → {END_STAGE}")

In [ ]:
# Cell 2b: (Optional) Override curriculum & opponents per stage
# Edit the dicts below, then re-run this cell before Cell 3.
import json, re
from pathlib import Path

ROOT = Path.cwd()
CUSTOM_CURRICULUM_PATH = None  # set by this cell

# Override max_steps per stage (set None to keep default):
CUSTOM_MAX_STEPS = {
    0: 1000,
    1: 1000,
}

# Override opponent pools per stage (builtin name or codex file path):
CUSTOM_OPPONENT_POOLS = {
    0: ["RandomAgent", "RandomAgent", "RandomAgent"],
}

# --- Apply ---
if CUSTOM_MAX_STEPS:
    base = ROOT / "configs" / "curriculum_stagewise_v2.json"
    cur = json.loads(base.read_text())
    for s, ms in CUSTOM_MAX_STEPS.items():
        if ms is not None:
            cur[str(s)]["max_steps"] = ms
    custom = ROOT / "configs" / "custom_curriculum.json"
    custom.write_text(json.dumps(cur, indent=2))
    CUSTOM_CURRICULUM_PATH = str(custom)
    print(f"Custom curriculum written to {custom}")

if CUSTOM_OPPONENT_POOLS:
    script = ROOT / "scripts" / "participant" / "train_bc_ppo.py"
    content = script.read_text()
    for s, pool in CUSTOM_OPPONENT_POOLS.items():
        pool_str = json.dumps(pool)
        pat = re.compile(r'^(\s+)' + str(s) + r':\s*\[.*?\],\s*$', re.MULTILINE)
        if pat.search(content):
            content = pat.sub(rf'\g<1>{s}: {pool_str},', content)
            print(f"Patched stage {s} opponents: {pool_str}")
    script.write_text(content)

if not CUSTOM_MAX_STEPS and not CUSTOM_OPPONENT_POOLS:
    print("No custom overrides set (dicts are empty).")

# Will be picked up by Cell 3 if not None


## Main Training

Runs BC pretraining then stage-wise PPO from stage 0 → 7.
If `SKIP_BC = True`, resumes from existing BC checkpoint.

Checkpoints are saved to `checkpoints/stagewise/`.

In [ ]:
# Cell 3: Run stage-wise BC + PPO training
train_flags = [
    sys.executable, "-u", "scripts/participant/stage_train_bc_ppo.py",
    "--profile", PROFILE,
    "--device", DEVICE,
    "--seed", str(SEED),
    "--eval-matches", str(EVAL_MATCHES),
    "--torch-threads", str(TORCH_THREADS),
    "--export-dir", EXPORT_DIR,
]

if SKIP_BC:
    train_flags.append("--skip-bc")
if START_STAGE > 0:
    train_flags.extend(["--start-stage", str(START_STAGE)])
if END_STAGE < 7:
    train_flags.extend(["--end-stage", str(END_STAGE)])
if "CUSTOM_CURRICULUM_PATH" in globals() and CUSTOM_CURRICULUM_PATH is not None:
    train_flags.extend(["--curriculum-config", CUSTOM_CURRICULUM_PATH])

print("Running:", " ".join(train_flags))
stream_run(train_flags)

## Export Agent

Exports the trained model + bundled codex agents into a standalone directory.
The exported `agent.py` uses a conservative neural override gate with teacher fallback.

In [ ]:
# Cell 4: Export the trained agent (if stage_train didn't already export)
LATEST_CHECKPOINT = Path("checkpoints/stagewise/stage7.pt")

if LATEST_CHECKPOINT.exists():
    !python -u scripts/participant/train_bc_ppo.py \
        --mode export \
        --device cpu \
        --checkpoint $LATEST_CHECKPOINT \
        --export-dir $EXPORT_DIR
    print(f"Exported to {EXPORT_DIR}")
else:
    print(f"Checkpoint {LATEST_CHECKPOINT} not found. Did training finish?")

## Benchmark

Benchmark the exported agent against codex agents `4.py`, `7.py`, `8.py`.
Uses 50 matches, 400 max steps with 100ms timeout enforcement.

In [ ]:
# Cell 5: Benchmark exported agent
EXPORT_PATH = Path(EXPORT_DIR)
if EXPORT_PATH.exists():
    !python -m scripts.participant.benchmark \
        --agents $EXPORT_DIR agent/codex/4.py agent/codex/7.py agent/codex/8.py \
        --matches 50 \
        --max_steps 400 \
        --timeout \
        --timeout-ms 100
else:
    print(f"Export directory {EXPORT_DIR} not found.")

## Select Best Checkpoint

Do **not** assume the final stage is best. Benchmark each stage checkpoint
and pick the one with the best win count, average rank, and timeout count.

The cell below exports and benchmarks each stage checkpoint automatically.

In [ ]:
# Cell 6: Benchmark individual stage checkpoints to select the best
from pathlib import Path
import subprocess, sys

CKPT_DIR = Path("checkpoints/stagewise")
if CKPT_DIR.exists():
    for stage in range(8):
        ckpt = CKPT_DIR / f"stage{stage}.pt"
        if not ckpt.exists():
            print(f"Stage {stage}: checkpoint not found, skipping")
            continue
        export_dir = f"exports/stage{stage}_agent"
        stream_run([
            sys.executable, "-u", "scripts/participant/train_bc_ppo.py",
            "--mode", "export", "--device", "cpu",
            "--checkpoint", str(ckpt), "--export-dir", export_dir,
        ])
        print(f"\n--- Benchmarking Stage {stage} ---")
        stream_run([
            sys.executable, "-m", "scripts.participant.benchmark",
            "--agents", export_dir, "agent/codex/4.py", "agent/codex/7.py", "agent/codex/8.py",
            "--matches", "50", "--max_steps", "400", "--timeout", "--timeout-ms", "100",
        ])
else:
    print("No checkpoints directory found.")

## (Optional) Late-Stage Finetune

If the main run beats `7.py` but still loses to `4.py`, continue training
stage 7 with more PPO updates. This cell loads the best stage-7 checkpoint
and runs additional PPO with a lower LR and higher eval matches.

In [ ]:
# Cell 7: (Optional) Late-stage finetune from best_stage7 checkpoint
BEST_CKPT = "checkpoints/stagewise/best_stage7.pt"
if Path(BEST_CKPT).exists():
    !python -u scripts/participant/train_bc_ppo.py \
        --mode ppo \
        --device $DEVICE \
        --checkpoint $BEST_CKPT \
        --save-checkpoint checkpoints/stagewise/late_stage_finetune.pt \
        --curriculum-config configs/curriculum_stagewise_v2.json \
        --fixed-stage 7 \
        --ppo-updates 600 \
        --ppo-envs-per-update 16 \
        --ppo-horizon 192 \
        --eval-interval 10 \
        --eval-matches 20 \
        --snapshot-interval 25 \
        --torch-threads $TORCH_THREADS \
        --export-dir exports/stagewise_late_finetune_agent
    print("Late-stage finetune complete.")
else:
    print(f"{BEST_CKPT} not found. Run the main training first.")

## Create Submission Zip

Package the best agent into a submission zip for Kaggle/Colab.

In [ ]:
# Cell 8: Create submission zip from exported agent
EXPORT_PATH = Path(EXPORT_DIR)
if EXPORT_PATH.exists():
    zip_name = EXPORT_PATH.name + "_submission.zip"
    !cd $EXPORT_DIR && zip -r ../../$zip_name agent.py model.pt 4.py 7.py 8.py
    print(f"Created {zip_name}")
else:
    print(f"Export directory {EXPORT_DIR} does not exist.")

## Notes

- Do **not** trust a checkpoint only because training reward improved. Select using benchmark + timeout.
- If neural-only/masked eval is weak, keep the conservative export gate (default). It behaves like a strong teacher-league agent with occasional learned overrides.
- If `fallback_rate` remains high through late stages, increase BC matches/epochs before increasing PPO.
- Prefer: `150 matches × 50 epochs`. Avoid `10 matches × 250 epochs` (overfits).